# 06 — Propensity Scores, IPW, and Doubly Robust Estimation
**References:** Rosenbaum & Rubin (1983, *Biometrika*) · Hernán & Robins (2020) Ch. 12–13, 15 · Robins, Rotnitzky & Zhao (1994) · Imbens & Rubin (2015) Ch. 12–14 · Stanford STATS 361 Lectures 2–3

## Narrative thread
```
The balancing property -> Estimating e(x) -> IPW & the Horvitz-Thompson idea -> Stabilized weights -> Doubly robust AIPW
```

## The propensity score theorem (Rosenbaum & Rubin 1983)

The **propensity score** is $e(x) = P(W = 1 \mid X = x)$. Two key results:

1. **Balancing:** $X \perp W \mid e(X)$ — units with the same propensity score have the
   same covariate distribution in both arms.
2. **Sufficiency:** if unconfoundedness holds given $X$, it holds given $e(X)$:
   $(Y(0), Y(1)) \perp W \mid e(X)$.

So the entire covariate vector can be collapsed to **one dimension** — defeating the curse
of dimensionality that plagues matching on raw $X$.

## Inverse probability weighting (IPW)

Weight each unit by the inverse probability of the treatment it actually received
(the Horvitz-Thompson idea from survey sampling):

$$\hat\tau_{IPW} = \frac{1}{n}\sum_i \frac{W_i Y_i}{\hat e(X_i)} - \frac{1}{n}\sum_i \frac{(1-W_i) Y_i}{1 - \hat e(X_i)}$$

Weighting creates a **pseudo-population** in which treatment is independent of $X$.
In practice use the **Hájek** (normalized) version — divide by the sum of weights instead
of $n$ — and **stabilized weights** to tame variance. Extreme scores ($e(x)$ near 0 or 1)
signal overlap problems: trimming to $[0.05, 0.95]$ or $[0.1, 0.9]$ is standard (Crump et
al. 2009).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── Simulated observational study used throughout this notebook ──────────
def make_data(n=20_000, seed=42):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(0, 1, n)
    x2 = rng.normal(0, 1, n)
    e  = 1 / (1 + np.exp(-(0.6*x1 - 0.8*x2)))            # true propensity
    w  = rng.binomial(1, e)
    y0 = 200 + 25*x1 + 12*x2**2 + rng.normal(0, 15, n)   # NONLINEAR outcome model
    y1 = y0 + 10                                          # true ATE = 10
    y  = np.where(w == 1, y1, y0)
    return pd.DataFrame(dict(x1=x1, x2=x2, w=w, y=y, e_true=e))

df = make_data()
naive = df.query('w==1').y.mean() - df.query('w==0').y.mean()
print(f'True ATE = 10.00,  naive difference = {naive:.2f}')

# ── The balancing property: X | e(X) is balanced across arms ─────────────
df['e_bin'] = pd.qcut(df['e_true'], 5, labels=False)
print('\nMean of x1 by treatment arm, within propensity-score quintiles:')
bal = df.groupby(['e_bin', 'w'], observed=True)['x1'].mean().unstack()
bal.columns = ['control', 'treated']
bal['gap'] = bal['treated'] - bal['control']
print(bal.round(3).to_string())
print('\nRaw gap in x1 (no stratification):'
      f' {df.query("w==1").x1.mean() - df.query("w==0").x1.mean():+.3f}')

In [ ]:
# ── IPW step by step, with an ESTIMATED propensity score ─────────────────
# 1. Fit e(x) by logistic regression
ps_model = smf.logit('w ~ x1 + x2', df).fit(disp=0)
df['e_hat'] = ps_model.predict(df)

# 2. Check overlap before trusting any weighted estimate
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(df.query('w==1').e_hat, bins=40, alpha=0.6, density=True, label='treated')
ax.hist(df.query('w==0').e_hat, bins=40, alpha=0.6, density=True, label='control')
ax.set_xlabel('estimated propensity score'); ax.set_title('Overlap check')
ax.legend(); plt.tight_layout(); plt.show()

# 3. Hajek (normalized) IPW estimate of the ATE
w_, y_, e_ = df.w.values, df.y.values, df.e_hat.values
wt1 = w_ / e_
wt0 = (1 - w_) / (1 - e_)
tau_ipw = np.sum(wt1 * y_) / np.sum(wt1) - np.sum(wt0 * y_) / np.sum(wt0)
print(f'IPW (Hajek) ATE estimate: {tau_ipw:.2f}   (true: 10.00)')
print(f'Weight extremes: max treated weight = {wt1.max():.1f}, '
      f'max control weight = {wt0.max():.1f}')

## Doubly robust estimation (AIPW)

IPW needs $\hat e(x)$ right; outcome regression (the g-formula) needs
$\hat\mu_w(x) = \hat E[Y \mid W{=}w, X{=}x]$ right. The **augmented IPW** estimator
combines them:

$$\hat\tau_{AIPW} = \frac{1}{n}\sum_i \left[
\hat\mu_1(X_i) - \hat\mu_0(X_i)
+ \frac{W_i\,(Y_i - \hat\mu_1(X_i))}{\hat e(X_i)}
- \frac{(1-W_i)\,(Y_i - \hat\mu_0(X_i))}{1 - \hat e(X_i)} \right]$$

**Double robustness** (Robins, Rotnitzky & Zhao 1994): $\hat\tau_{AIPW}$ is consistent if
*either* the propensity model *or* the outcome model is correctly specified — two chances
to get it right. When both are right it attains the semiparametric efficiency bound; this
same functional is the foundation of double machine learning (notebook 11).


In [ ]:
# ── Double robustness demonstrated: 2 models x {right, wrong} ────────────
from sklearn.linear_model import LinearRegression, LogisticRegression

def aipw(df, e_hat, mu1, mu0):
    w_, y_ = df.w.values, df.y.values
    return np.mean(mu1 - mu0
                   + w_ * (y_ - mu1) / e_hat
                   - (1 - w_) * (y_ - mu0) / (1 - e_hat))

def fit_models(df, ps_right, out_right):
    # Propensity: right uses (x1, x2); wrong omits x2 entirely
    ps_feats = ['x1', 'x2'] if ps_right else ['x1']
    e_hat = LogisticRegression(C=1e6).fit(df[ps_feats], df.w).predict_proba(df[ps_feats])[:, 1]
    e_hat = np.clip(e_hat, 0.01, 0.99)
    # Outcome: right uses (x1, x2^2); wrong uses linear x1 only
    if out_right:
        Xo = np.column_stack([df.x1, df.x2**2])
    else:
        Xo = df[['x1']].values
    mu = {}
    for arm in (0, 1):
        m = LinearRegression().fit(Xo[df.w == arm], df.y[df.w == arm])
        mu[arm] = m.predict(Xo)
    return e_hat, mu[1], mu[0]

reps = 200
print(f'{"PS model":<10} {"outcome model":<15} {"mean AIPW":>10} {"bias":>8}')
for ps_right in (True, False):
    for out_right in (True, False):
        ests = []
        for r in range(reps):
            d = make_data(n=2000, seed=1000 + r)
            ests.append(aipw(d, *fit_models(d, ps_right, out_right)))
        m = np.mean(ests)
        tag_ps  = 'right' if ps_right else 'WRONG'
        tag_out = 'right' if out_right else 'WRONG'
        print(f'{tag_ps:<10} {tag_out:<15} {m:10.2f} {m-10:8.2f}')
print('\nAIPW is biased only when BOTH nuisance models are wrong (true ATE = 10).')

## Choosing among the estimators

| Estimator | Needs correct | Pros | Cons |
|---|---|---|---|
| Outcome regression (g-formula) | $\mu_w(x)$ | Efficient if right | Silent extrapolation off-support |
| PS matching / stratification | $e(x)$ | Transparent, design-based | Residual bias, awkward inference |
| IPW (Hájek, stabilized) | $e(x)$ | Simple, honest about overlap | High variance with extreme weights |
| **AIPW** | either one | Doubly robust, efficient | More moving parts |

Hernán & Robins' practical advice: always inspect the distribution of $\hat e(X)$ first —
no estimator rescues a study without overlap.

## Key takeaways

| Concept | Statement |
|---|---|
| Propensity score | $e(x) = P(W{=}1 \mid x)$ — the coarsest balancing score |
| Balancing property | $X \perp W \mid e(X)$: 1-D summary of all confounders |
| IPW | Reweight to a pseudo-population where treatment ⟂ covariates |
| Stabilized/Hájek weights | Normalize; trim extreme scores; check overlap always |
| AIPW | Outcome model + IPW correction; consistent if either is right |
| Efficiency | AIPW attains the semiparametric bound when both models are right |
